# alternating soundsource analysis

In [31]:
from workbench.data.preprocess import  combTableCreate, expand_dict_columns
import numpy as np
import pandas as pd
import polars as pl
import sqlite3
from pathlib import Path
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
import seaborn as sns

In [32]:
conn = sqlite3.connect(r"\\172.25.250.112\burgalossi\lab share\Data\Florian\Recordings_FH.db")
sql = """
SELECT * FROM Recordings WHERE (Animal_Id, Cell_Id) IN (
SELECT Animal_Id, Cell_Id FROM Recordings WHERE Condition IN ("Baseline", "soso") GROUP BY Animal_Id, Cell_Id 
HAVING COUNT(DISTINCT Condition) >= 2) 
AND Condition IN ("Baseline","soso") AND use = 1 AND Folders_generated = 1
"""

datatable= pd.read_sql_query(sql, conn)
conn.close()

In [33]:
try:
    comb_table = pl.read_parquet(Path(r"\\172.25.250.112\burgalossi\lab share\Data\Florian\comb_tables\soso_comb.parquet"))
except:
    print('comb_table doesnt exist, start creation')
    comb_table = combTableCreate(datatable, "", "")
    comb_df = pd.DataFrame(comb_table)
    dict_list = [ 'pupil_psth', 'whisk_psth', 'eye_psth']
    comb_df = expand_dict_columns(
        comb_df,
        dict_columns=dict_list,
        flatten_2d=False
    )
    # clean and reshape the dataframe
    comb_table = pl.from_dataframe(comb_df)
    conditions = comb_table['Condition'].unique().to_list()

    comb_a = (
        comb_table.filter(pl.col("Condition") == conditions[0]).drop("Condition")
    )
    comb_b = (
        comb_table.filter(pl.col("Condition") == conditions[1]).drop("Condition")
    )

    comb_joined = comb_a.join(
        comb_b,
        on=['Animal_Id', 'Cell_Id'],
        how="inner"
    )

    comb_joined = comb_joined[[s.name for s in comb_joined if not (s.null_count() == comb_joined.height)]]
    comb_table = comb_joined.rename(lambda c: c[:-6] if c.endswith("_right") else c)
    comb_table.write_parquet(r"Z:\lab share\Data\Florian\comb_tables\soso_comb.parquet", use_pyarrow=True)
    print("table created and saved")

In [34]:
# analysis
speaker_position = {
    'a': 93,
    'w': 178,
    'e': 272,
    'r': 356
}
rads_x = np.cos(np.deg2rad(list(speaker_position.values())))
rads_y = np.sin(np.deg2rad(list(speaker_position.values())))
#returned_degs = np.degrees(np.arctan2(rads_y, rads_x))
#returned_degs = (returned_degs+360)%360


For the analysis i should start with per recording representation of activity per speaker position. What about Rasterplots stacked on top of each other (4), each corresponding to a different speaker.
For now this will happen for all recordings individually before analysis across all follows

In [ ]:
# cell by cell sound source representation

half_window = 1500 # in ms
time_bin = 0.002
nbins = int(np.round(half_window/(time_bin*1000)))
raster_edges = np.linspace(-half_window, half_window, nbins*2+1)

# mimic the medfilt1
bins_plot = np.concatenate(
    [[raster_edges[0]],
      np.median(
          np.vstack( # create a 2d array with the raster edges shifted by 1 each. the median then picks the value right in between them
              [raster_edges[:-1],
                raster_edges[1:]]
                )
            , axis=0)
    ]
)
bins_plot = bins_plot[1::]

raster_times = comb_table['RasterTimes']
raster_rows = comb_table['RasterRows']
raster_rate = comb_table['RasterRate']
HDRateSmooth = comb_table['hdRateSmooth']
DIRECTIONS = np.linspace(0, 360, 37)
stim_keys = raster_times[0].keys()
trigger_time = comb_table['trigger_time'][0]
whisk_avg = comb_table['whisk_avg']
new_rate = np.zeros((len(raster_times), len(raster_edges)-1, len(keys)))

for i in range(len(raster_times)):
    for idx, j in enumerate(keys):
        if not len(raster_times[i][j]) == 0:
            counts, _ = np.histogram(raster_times[i][j], bins=raster_edges)
            new_rate[i, :, idx] = np.divide(counts, (np.max(raster_rows[i][j])*time_bin))

# construct whisking
#with PdfPages(r"\\172.25.250.112\burgalossi\lab share\Data\Florian\soundsource_switching\individuals.pdf") as pdf:
speakers = ["a", "w", "e", "r"]
speaker_position = {
    "a": 93,
    "w": 178,
    "e": 272,
    "r": 356,
}

# Your mosaic layout (4 rows × 4 columns)
layout = [
    ["r_a", "psth_a", "line_a", "polar"],
    ["r_w", "psth_w", "line_w", "."    ],
    ["r_e", "psth_e", "line_e", "."    ],
    ["r_r", "psth_r", "line_r", "."    ],
]


# ─────────────────────────────────────────────────────────────
#  MAIN LOOP: one page per cell
# ─────────────────────────────────────────────────────────────
for row_idx in range(len(raster_rate)):

    fig, axd = plt.subplot_mosaic(
        layout,
        figsize=(15, 12),
        gridspec_kw={"width_ratios": [2.5, 3, 3, 2.5]},
        empty_sentinel=".",       # treat "." cells as empty
    )

    # ----------------------------------------------------------
    # Convert the placeholder "polar" axis into a real polar axes
    # ----------------------------------------------------------
    polar_spec = axd["polar"].get_subplotspec()
    axd["polar"].remove()
    ax_polar = fig.add_subplot(polar_spec, projection="polar")

    # ----------------------------------------------------------
    # Loop over speakers and fill rasters, PSTHs, lineplots
    # ----------------------------------------------------------

    tmin, tmax = -50, 100   # example x-range (adjust as needed)

    for enum, key in enumerate(speakers):
        # Named axes from the mosaic
        ax_r = axd[f"r_{key}"]
        ax_p = axd[f"psth_{key}"]
        ax_l = axd[f"line_{key}"]

        # Set x-limits for all time-based axes
        for ax in (ax_r, ax_p, ax_l):
            ax.set_xlim(tmin, tmax)

        # ---------------- RASTER PLOT ----------------
        ax_r.scatter(
            raster_times[row_idx][key],
            raster_rows[row_idx][key],
            marker=".",
        )
        ax_r.set_ylabel(speaker_position[key])

        # ---------------- PSTH PLOT ------------------
        ax_p.bar(bins_plot, new_rate[row_idx, :, enum], width=time_bin*1000, align='edge', edgecolor='none')
        ax_p.set_ylabel("Firing Rate [Hz]")

        # ---------------- LINE PLOT ------------------
        ax_l.plot(trigger_time*1000, whisk_avg[row_idx][key])

    # ----------------------------------------------------------
    # Despine + tidy Cartesian axes
    # ----------------------------------------------------------

    def despine(ax):
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

    for key in speakers:
        despine(axd[f"r_{key}"])
        despine(axd[f"psth_{key}"])
        despine(axd[f"line_{key}"])

        # Cleaner PSTH/line look (optional):
        #axd[f"psth_{key}"].tick_params(axis="y", left=False, labelleft=False)
        #axd[f"line_{key}"].tick_params(axis="y", left=False, labelleft=False)

    # ----------------------------------------------------------
    # POLAR PLOT: mark speaker angles & plot tuning (optional)
    # ----------------------------------------------------------

    angles_deg = list(speaker_position.values())
    labels     = list(speaker_position.keys())
    angles_rad = np.deg2rad(angles_deg)

    # Angle grid with labels a/w/e/r
    ax_polar.set_thetagrids(angles_deg, labels)
    ax_polar.plot(np.deg2rad(DIRECTIONS), HDRateSmooth[row_idx])

    # Optional: put markers at those angles (radius = 1)
    ax_polar.scatter(angles_rad, np.ones(len(angles_rad)), s=40)

    # Optional tuning curve:
    # rate = np.array([...])   # length 4, matching angles_deg
    # ax_polar.plot(angles_rad, rate)

    # ----------------------------------------------------------
    # Final layout touch
    # ----------------------------------------------------------
    fig.tight_layout()

    # If saving to PDF:
    # pdf.savefig(fig)
    # plt.close(fig)

plt.show()